# Lab 12e : Persistance d'état de session - une conversation qui se souvient

**Navigation** : [Lab 12d <<](Lab12d-Token-Usage.ipynb) | [README du track](../README.md)

Ce lab porte le **contrat comportemental C1b** du corpus EPITA (#14058) : *l'historique
d'un tour doit atteindre le tour suivant, au sein d'une même conversation*.

Dans les labs précédents, chaque appel au runtime (`run_agent_turn`) créait une session
**fraîche** : l'agent répondait à chaque tour comme s'il découvrait la conversation. Nous
allons d'abord **mesurer** ce non-portage avec le LLM réel, puis porter le contrat
**au-dessus d'ADK** avec `ConversationRunner` (`utils/adk_conversation.py`) : un seul
service de session et un seul identifiant de session partagé par tous les tours d'une
conversation.

**Ce que vous saurez faire à la fin** :
- démontrer pourquoi deux tours isolés ne partagent aucun état ;
- maintenir une conversation multi-tours persistante sur le runtime ADK réel ;
- prouver mécaniquement (par le contenu de la session) que la mémoire existe ;
- garder l'isolation entre conversations distinctes (contrat C1) tout en portant C1b.

## 1. Configuration

Le track charge son provider LLM depuis la configuration (`ACTIVE_PROVIDER` dans le `.env`
du track). Le runtime ADK reste le même qu'aux labs 11-12 : agent ADK réel, runner réel,
session ADK réelle. Aucune réponse n'est simulée.

In [1]:
import sys
sys.path.insert(0, '..')

import warnings

warnings.filterwarnings(
    "ignore",
    message=(
        r"\[EXPERIMENTAL\] feature "
        r"FeatureName\.JSON_SCHEMA_FOR_FUNC_DECL is enabled\."
    ),
    category=UserWarning,
    module=r"google\.adk\.models\.llm_request",
)

from config.providers import get_settings, get_provider_config, get_litellm_model

settings = get_settings()
provider = get_provider_config(settings)
print(f"Provider actif : {provider.provider.value}")
print(f"Modele : {get_litellm_model(provider)}")
print(f"Endpoint externe : {bool(provider.base_url)}")

Provider actif : openrouter
Modele : openrouter/openai/gpt-5.6-luna
Endpoint externe : True


**Lecture** : Verbatim : « Provider actif : openrouter », « Modele : openrouter/openai/gpt-5.6-luna », « Endpoint externe : True » — un modèle léger via passerelle pour ce lab. Le choix du modèle ne change rien à la question de la persistance de session ; seul le compte de jetons (Lab12d) en dépend.

Le provider actif est celui du `.env` du track (vLLM local, OpenRouter ou Gemini selon
la machine). Le lab ne dépend pas du modèle choisi : le contrat C1b teste la mécanique
de session du runtime, pas la qualité des réponses du LLM.

## 2. Le problème : deux tours isolés ne partagent rien

Mesurons le comportement par défaut du runtime. Nous posons **deux questions séparées**
à l'agent : la première établit un contexte métier (un dataset précis), la seconde
demande un **rappel** de ce contexte -- sans jamais le répéter dans la question.

In [2]:
from utils.adk_runtime import run_agent_turn, build_data_agent

agent = build_data_agent()

# Tour 1 : le contexte metier est pose.
tour1 = await run_agent_turn(
    agent,
    "Je travaille sur le dataset 'patients-2026' : 240 lignes et 12 colonnes. "
    "Appelle dataset_profile pour ce dataset et resume le resultat.",
    timeout_seconds=90,
)
print("TOUR 1 -- reponse :")
print(tour1.response_text)

TOUR 1 -- reponse :
Le dataset **« patients-2026 »** contient :

- **240 lignes**
- **12 colonnes**
- **2 880 cellules** au total
- **20 lignes par colonne** en moyenne

Il s’agit donc d’un jeu de données tabulaire de taille modérée.


**Lecture** : Réponse du tour 1, reformulée : le dataset « patients-2026 » compte 240 enregistrements et 12 colonnes, soit 2 880 valeurs au total, et 20 valeurs par colonne en moyenne — un jeu de données tabulaire de taille modérée. Le détail à retenir : l'agent ne recopie pas l'énoncé — il a appelé `dataset_profile` et calculé la multiplication (240 × 12 = 2 880) et le rapport enregistrements/colonnes (20 valeurs par colonne en moyenne). Le contexte posé au tour 1 est plus riche que la question : c'est ce contenu-là que le tour 2 devra retrouver.

In [3]:
# Tour 2 : rappel demande SANS repeter le contexte.
tour2 = await run_agent_turn(
    agent,
    "Combien de lignes et de colonnes avait mon dataset, et comment s'appelait-il ?",
    timeout_seconds=90,
)
print("TOUR 2 -- reponse :")
print(tour2.response_text)

TOUR 2 -- reponse :
Je n’ai pas accès aux informations de votre dataset dans cette conversation. Veuillez fournir son nom ainsi que le nombre de lignes et de colonnes, et je pourrai vous le confirmer.


### Lecture du résultat

La réponse du tour 2 est celle d'un agent **amnésique** : « Je n’ai pas accès aux informations de votre dataset dans cette conversation. Veuillez fournir son nom ainsi que le nombre de lignes et de colonnes, et je pourrai vous le confirmer. » -- il ne connait ni le nom `patients-2026`,
ni les dimensions de son dataset, il n'a jamais recu cette information. Ce n'est pas un defaut du
modèle : c'est le runtime qui, à chaque appel à `run_agent_turn`, instancie un
`InMemorySessionService` neuf et une session neuve. Le message du tour 1 n'existe
simplement plus au tour 2.

C'est la mesure du **non-portage du contrat C1b** -- le même constat que celui consigné
par le test `test_state_does_not_persist_across_calls` dans
`utils/test_adk_runtime_contracts.py`.

## 3. La solution au-dessus d'ADK : `ConversationRunner`

ADK porte nativement la mécanique qu'il nous faut : une **session est conçue pour
durer**, et le runner y accumule l'historique à chaque `run_async`. Le non-portage venait
de l'assemblage (tout recréer par appel), pas du moteur.

`utils/adk_conversation.py` fait l'assemblage durable : **un seul** service de session,
**un seul** `session_id`, un runner pour toute la conversation. Aucune logique propre --
uniquement les primitives publiques d'ADK.

In [4]:
from utils.adk_conversation import ConversationRunner

print(ConversationRunner.__doc__.strip()[:600])
print("...")
print("Deux attributs font tout le contrat :")
print("  - self.session_service : un seul InMemorySessionService pour TOUS les tours")
print("  - self.session_id      : identifiant stable, jamais regenere entre les tours")

Execute plusieurs tours d'un agent dans la MEME session ADK.

La conversation detient un ``InMemorySessionService`` et un ``session_id``
uniques : chaque tour herite de l'historique des tours precedents, ce qui
porte le contrat C1b (persistance d'etat entre tours). Le cloisonnement
reste celui du contrat C1 : deux ``ConversationRunner`` sont deux sessions
distinctes, isolees l'une de l'autre.

``budget_total_tokens`` (contrat C6, jambe budget) pose un plafond sur le
cumul de jetons de la conversation : un tour est refuse AVANT tout appel
LLM quand le plafond est deja atteint, et coupe au premi
...
Deux attributs font tout le contrat :
  - self.session_service : un seul InMemorySessionService pour TOUS les tours
  - self.session_id      : identifiant stable, jamais regenere entre les tours


**Lecture** : Le docstring porte les deux contrats d'une phrase : « Execute plusieurs tours d'un agent dans la MEME session ADK » (C1b) et « Le cloisonnement
reste celui du contrat C1 : deux ``ConversationRunner`` sont deux sessions
distinctes, isolees l'une de l'autre ». Les deux attributs imprimés font tout le travail : « self.session_service : un seul InMemorySessionService pour TOUS les tours », « self.session_id      : identifiant stable, jamais regenere entre les tours ». Le docstring ajoute le contrat C6 : `budget_total_tokens` plafonne le cumul de jetons de la conversation, et un tour est refusé AVANT tout appel LLM quand le plafond est atteint. Aucune logique propre — l'assemblage réutilise les primitives publiques d'ADK ; ce qui manquait au tour isolé de la section 2, c'était uniquement la durée de vie de ces deux objets.

La conversation va maintenant jouer **trois tours** sur la même session : le tour 1 pose
le contexte, le tour 2 demande le rappel, le tour 3 demande un rappel plus profond
(une valeur **calculée** au tour 1 par l'outil, pas seulement récitée).

In [5]:
conv_agent = build_data_agent()

async with ConversationRunner(conv_agent) as conv:
    t1 = await conv.turn(
        "Je travaille sur le dataset 'patients-2026' : 240 lignes et 12 colonnes. "
        "Appelle dataset_profile et resume le resultat.",
        timeout_seconds=90,
    )
    t2 = await conv.turn(
        "Combien de lignes et de colonnes avait mon dataset, et comment s'appelait-il ?",
        timeout_seconds=90,
    )
    t3 = await conv.turn(
        "Et combien de cellules au total (recalcule si besoin) ?",
        timeout_seconds=90,
    )
    history = await conv.history()
print("TOUR 2 -- reponse (rappel du contexte) :")
print(t2.response_text)
print()
print("TOUR 3 -- reponse (rappel de la valeur calculee) :")
print(t3.response_text)

TOUR 2 -- reponse (rappel du contexte) :
Votre dataset s’appelait **« patients-2026 »** et contenait **240 lignes** et **12 colonnes**.

TOUR 3 -- reponse (rappel de la valeur calculee) :
Le dataset contenait **2 880 cellules au total** :

**240 lignes × 12 colonnes = 2 880 cellules**.


### Lecture du résultat

Le tour 2 rappelle le nom et la forme du dataset ; le tour 3 retrouve la valeur calculée
(2 880 valeurs = 240 x 12). L'agent n'a **jamais** recu ces informations au tour 2 ou 3 :
elles lui parviennent par l'historique de session qu'ADK attache à chaque requête LLM.
Le contrat C1b est porté -- par l'assemblage durable des primitives ADK, pas par une
restauration du moteur SK (#14058).

## 4. Preuve mécanique : la session cumule les tours

La mémoire ne doit pas être un effet de bord du modèle : elle doit être **observable dans
la session**. Un événement ADK par message -- après trois tours d'une conversation, la
session doit contenir l'historique complet (questions et réponses, appels d'outil
inclus).

In [6]:
print(f"Evenements persistes dans la session apres 3 tours : {len(history)}")
print()
print("Chronologie de la conversation (role : extrait) :")
for i, event in enumerate(history, start=1):
    if event.content and event.content.parts:
        texte = "".join(p.text or "" for p in event.content.parts).strip()
        if texte:
            print(f"  {i:2d}. [{event.author or '?':>8}] {texte[:95]}")
        elif event.get_function_calls():
            print(f"  {i:2d}. [{event.author or '?':>8}] <appel d'outil>")

Evenements persistes dans la session apres 3 tours : 8

Chronologie de la conversation (role : extrait) :
   1. [    user] Je travaille sur le dataset 'patients-2026' : 240 lignes et 12 colonnes. Appelle dataset_profil
   2. [dataset_profile_agent] <appel d'outil>
   4. [dataset_profile_agent] Le dataset **« patients-2026 »** contient :

- **240 lignes**
- **12 colonnes**
- **2 880 cellu
   5. [    user] Combien de lignes et de colonnes avait mon dataset, et comment s'appelait-il ?
   6. [dataset_profile_agent] Votre dataset s’appelait **« patients-2026 »** et contenait **240 lignes** et **12 colonnes**.
   7. [    user] Et combien de cellules au total (recalcule si besoin) ?
   8. [dataset_profile_agent] Le dataset contenait **2 880 cellules au total** :

**240 lignes × 12 colonnes = 2 880 cellules


### Lecture du résultat

La session détient la conversation complète : les trois questions utilisateur, les
réponses du modèle et les allers-retours d'outil. La chronologie imprimée le montre
événement par événement -- question en 1, appel d'outil en 2, profil en 4, rappel du
nom en 6, calcul en 8 (l'événement 3, masqué par le filtre d'impression, compte dans
les 8). C'est exactement ce contenu qu'ADK renvoie au LLM comme contexte au tour
suivant -- la preuve mécanique demandée par le critère 4 de #14058 : le test
`test_c1b_history_reaches_next_turn` vérifie que le marqueur du tour 1 figure dans
la requête LLM du tour 2, et échoue si la persistance est retirée.

## 5. Persistance ET isolation : C1b ne casse pas C1

Porter la mémoire **intra**-conversation ne doit pas créer de fuite **inter**-conversations.
Le contrat C1 (isolation) reste porté : deux `ConversationRunner` sont deux sessions ADK
distinctes. Vérifions qu'un contexte posé dans la conversation A est invisible depuis la
conversation B.

In [7]:
async def deux_conversations():
    # Conversation A : contexte exclusif.
    agent_a = build_data_agent()
    async with ConversationRunner(agent_a) as conv_a:
        await conv_a.turn(
            "Retiens : le dataset 'secret-alpha' fait 88 lignes et 4 colonnes.",
            timeout_seconds=90,
        )
        hist_a = await conv_a.history()
    # Conversation B : question SANS le nom -- le marqueur ne peut arriver
    # dans l'historique de B que par une fuite de session.
    agent_b = build_data_agent()
    async with ConversationRunner(agent_b) as conv_b:
        t = await conv_b.turn(
            "Combien de lignes avait MON dataset, et comment s'appelait-il ?",
            timeout_seconds=90,
        )
        hist_b = await conv_b.history()
    return hist_a, hist_b, t

hist_a, hist_b, reponse_b = await deux_conversations()
textes_a = "".join(
    (p.text or "") for e in hist_a for p in ((e.content and e.content.parts) or []))
textes_b = "".join(
    (p.text or "") for e in hist_b for p in ((e.content and e.content.parts) or []))
print("A contient 'secret-alpha'          :", "secret-alpha" in textes_a)
print("B contient 'secret-alpha' (fuite ?):", "secret-alpha" in textes_b)
print()
print("Reponse de B (attendue : aucun contexte recu) :")
print(reponse_b.response_text[:300])

A contient 'secret-alpha'          : True
B contient 'secret-alpha' (fuite ?): False

Reponse de B (attendue : aucun contexte recu) :
Je n’ai pas accès aux informations de votre dataset dans cette conversation. Aucun nom ni nombre de lignes n’a été fourni.


### Lecture du résultat

La conversation B ne contient mécaniquement aucune trace du contexte de A : sa
réponse, « Je n’ai pas accès aux informations de votre dataset dans cette conversation. Aucun nom ni nombre de lignes n’a été fourni. »,
redemande les dimensions du dataset -- l'isolation C1 tient. Les deux contrats
ensemble dessinent le **périmètre exact de l'état partagé** : partage au sein d'une
conversation, cloisonnement entre conversations. C'est la paire C1 + C1b, vérifiée par les tests `test_c1b_history_reaches_next_turn` et
`test_c1b_conversations_still_isolate`.

Le test croisé le confirme ligne à ligne : « A contient 'secret-alpha'          : True »,
« B contient 'secret-alpha' (fuite ?): False ».

## 6. Exercices

Trois exercices pour vous approprier le contrat. Le notebook doit rester executable de
bout en bout : les stubs ne levent jamais d'erreur.

### Exercice 1 -- Amnésie contrôlée

Certaines conversations doivent **oublier** : données sensibles, changement de client,
test isolé. Écrivez `oublie_et_redemarre` qui ferme la conversation courante et en rouvre
une neuve, puis vérifie que le nouveau tour ignore le contexte antérieur.

*Indice : il suffit de créer un nouveau `ConversationRunner` -- l'isolation C1 fait le
reste. Comparez les `history()` avant et après.*

In [8]:
# Exercice 1 : a completer
# Etape 1 : poser un contexte marqueur dans une conversation.
# Etape 2 : rouvrir une conversation neuve avec le meme agent.
# Etape 3 : demander le rappel du marqueur et afficher la reponse.

async def oublie_et_redemarre(agent, marqueur):
    """Renvoie (historique_avant, reponse_apres_oubli).

    La conversation initiale pose le marqueur ; une conversation neuve
    tente de le rappeler. L'historique de la premiere ne doit jamais
    contaminer la seconde.
    """
    # TODO etudiant : implementer avec deux ConversationRunner
    print("Exercice a completer")
    return None, None

# resultats = await oublie_et_redemarre(build_data_agent(), "memoire-vive-42")
# print(resultats)

### Exercice 2 -- Compteur d'événements par tour

Écrivez `evenements_par_tour` qui joue N tours sur une même conversation et renvoie la
liste des nombres d'événements persistés **après chaque tour**. La séquence attendue est
croissante : chaque tour ajoute ses événements à l'historique.

*Indice : appelez `history()` entre les tours, sans fermer la conversation.*

In [9]:
# Exercice 2 : a completer
# Etape 1 : ouvrir une ConversationRunner.
# Etape 2 : apres chaque tour, relever len(history()).
# Etape 3 : renvoyer la liste croissante.

async def evenements_par_tour(agent, invites):
    """Renvoie [len(history) apres le tour 1, apres le tour 2, ...]."""
    # TODO etudiant : implementer
    print("Exercice a completer")
    return None

# compteurs = await evenements_par_tour(build_data_agent(), ["tour un", "tour deux"])
# print("Compteurs :", compteurs)

### Exercice 3 -- Détecteur de fuite

Écrivez `fuite_entre_conversations` qui pose un marqueur dans la conversation A, fait
parler la conversation B, et renvoie `True` si le marqueur apparaît mécaniquement dans
l'historique de B (une fuite), `False` sinon. C'est le garde du contrat C1.

*Indice : la vérification est mécanique (parcourir `history()` de B), pas
interprétative -- ne demandez pas au LLM s'il se souvient, lisez la session.*

In [10]:
# Exercice 3 : a completer
# Etape 1 : conversation A pose un marqueur unique.
# Etape 2 : conversation B joue un tour quelconque.
# Etape 3 : chercher le marqueur dans history() de B et renvoyer le booleen.

async def fuite_entre_conversations(agent_a, agent_b, marqueur):
    """True si le marqueur de A atteint mecaniquement l'historique de B."""
    # TODO etudiant : implementer
    print("Exercice a completer")
    return False

# fuite = await fuite_entre_conversations(build_data_agent(), build_data_agent(), "fuite-cle-77")
# print("Fuite detectee :", fuite)  # attendu : False

## 7. Conclusion

| Idée clé | Où la voir |
|---|---|
| Deux tours isolés ne partagent aucun état | section 2, réponse amnésique du tour 2 |
| La mémoire vit dans la **session ADK**, pas dans le modèle | section 4, `history()` cumule |
| Porter C1b = assembler durablement les primitives ADK | `utils/adk_conversation.py` |
| C1b ne casse pas C1 : conversations toujours isolees | section 5 |
| Le contrat est gardé par un test qui échoue au retrait | `test_c1b_history_reaches_next_turn` |

**Où va le track ensuite** : les contrats C4 (désignation), C5 (handoff) et C6 (budgets
de jetons) restent non portés par le runtime du dépôt -- chacun est un grain ouvert
(#14058), mesuré dans `utils/test_adk_runtime_contracts.py`. Le transfert natif d'ADK
(`TransferToAgentTool`) et l'ordonnanceur dynamique (`workflow`) sont les primitives
candidates pour les tranches suivantes.

## À retenir

1. Deux tours isolés ne partagent rien : au tour 1 l'agent répond que le dataset compte 240 enregistrements et 12 colonnes, soit 2 880 valeurs ; au tour 2 isolé il répond « Je n’ai pas accès aux informations de votre dataset dans cette conversation. Veuillez fournir son nom ainsi que le nombre de lignes et de colonnes, et je pourrai vous le confirmer. ». La mémoire est perdue d'un tour à l'autre.
2. `ConversationRunner` porte le contrat C1b : une seule session ADK (`InMemorySessionService` et `session_id` uniques), chaque tour héritant de l'historique. Le tour 2 rappelle alors le nom du dataset (« patients-2026 ») et sa dimension — 240 × 12 (enregistrements × colonnes) — et le tour 3 recalcule le total à partir d'elle, « Votre dataset s’appelait **« patients-2026 »** et contenait **240 lignes** et **12 colonnes**. ».
3. La persistance est mesurée, pas affirmée : 8 événements persistent dans la session après 3 tours, et la chronologie complète (rôle et extrait) est lisible — 8 entrées pour 3 tours, parce que les appels d'outils sont eux aussi conservés.
4. Persister n'est pas fuir : le test de cloisonnement donne « A contient 'secret-alpha'          : True » et « B contient 'secret-alpha' (fuite ?): False » — C1b n'a pas cassé C1.

## Vérifiez votre compréhension

1. Que faut-il pour que le tour 3 retrouve la valeur calculée au tour 1 ?
2. Combien d'événements la session conserve-t-elle après 3 tours, et pourquoi ce n'est pas 3 ?
3. Comment vérifier qu'un marqueur posé dans A ne fuit pas vers B ?

<details><summary>Réponses</summary>

1. Deux choses, toutes deux portées par `ConversationRunner` : le même `session_service` (`InMemorySessionService`) et le même `session_id`, jamais regénéré entre les tours. C'est l'historique accumulé qui est réinjecté au tour 3, et c'est lui qui permet de recalculer « 240 lignes × 12 colonnes = 2 880 cellules » sans que rien soit redonné dans la question.
2. Huit événements. Les tours de l'utilisateur et de l'agent sont conservés, mais les appels d'outils aussi — c'est pourquoi le total dépasse le nombre de tours. Et la chronologie affichée le montre : on y voit les demandes de l'utilisateur, les appels d'outils et les réponses de l'agent dans l'ordre.
3. Par le marqueur de cloisonnement. On place une chaîne sentinelle dans A (`secret-alpha`), on interroge B, puis on cherche cette chaîne dans ce que B a reçu : ici « A contient 'secret-alpha'          : True » et « B contient 'secret-alpha' (fuite ?): False ». Un test de fuite se formule toujours comme la recherche d'un marqueur connu, jamais comme une impression.

</details>

## Pour aller plus loin

- **Carnet précédent** : [Lab12d-Token-Usage.ipynb](Lab12d-Token-Usage.ipynb) — C6, la consommation rendue observable : la persistance se paie à chaque tour.
- **Contrats voisins** : [Lab12c-Agent-Handoff.ipynb](Lab12c-Agent-Handoff.ipynb) (C5, le transfert décidé pendant le tour) et [Lab12b-Sequential-Orchestration.ipynb](Lab12b-Sequential-Orchestration.ipynb) (C4, l'ordre posé par l'appelant).
- **Index** : [README du track](../README.md) · [README de la série ML](../../README.md).
- **Lecture** : Nam et al., *DS-STAR: Data Science Agent for Solving Diverse Tasks across Heterogeneous Formats and Open-Ended Queries*, arXiv:2509.21825, 2025 ; Xi et al., *The Rise and Potential of Large Language Model Based Agents: A Survey*, arXiv:2309.07864, 2023.